# Notebook 04: Sheet Dynamics and Constraint Relaxation

This notebook adds **pseudo-time evolution** to the fragmented current-sheet model.

Notebook 03 gave:

- a sheet-localized fragmented band
- pocketed below-threshold regions
- static 45° contour structure

Notebook 04 asks:

- what happens when the fragmented sheet is allowed to evolve?
- do the pockets persist, merge, sharpen, or relax?

This is **not** a full MHD solver. It is a controlled dynamics model with:

- diffusion for smoothing
- restoring drift toward a Harris-sheet baseline
- activity near the 45° gate
- damping of the transverse field

Core geometric gate:

\[
\cos\theta \ge \frac{1}{\sqrt{1^2 + 1^2}}
\]

Interpretation:

- above threshold: broadly aligned field
- below threshold: strong-shear region
- activity near the threshold drives local evolution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

np.random.seed(21)

THRESHOLD = 1 / np.sqrt(2)

nx, ny = 360, 280
x = np.linspace(-6.0, 6.0, nx)
y = np.linspace(-3.0, 3.0, ny)
X, Y = np.meshgrid(x, y)

dx = x[1] - x[0]
dy = y[1] - y[0]

print("45° threshold =", THRESHOLD)
print("grid shape =", X.shape)
print("dx, dy =", dx, dy)

## 1. Helpers

We reuse the same basic field geometry as Notebook 03 and add a discrete Laplacian for pseudo-time evolution.

In [ ]:
def normalize_field(Bx, By):
    mag = np.sqrt(Bx**2 + By**2)
    mag = np.where(mag == 0, 1.0, mag)
    return Bx / mag, By / mag

def multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=0.38):
    out = np.zeros_like(X)
    envelope = np.exp(-(Y**2) / (sigma**2))
    for a, k, p in zip(amps, ks, phases):
        out += a * np.sin(k * X + p)
    return envelope * out

def harris_target(X, Y, delta=0.35):
    return np.tanh(Y / delta)

def initial_field(X, Y, delta=0.35, base_eps=0.06, frag_amp=0.28, sigma=0.38):
    Bx = np.tanh(Y / delta)

    By_background = base_eps * np.sin(1.2 * X) * np.exp(-(Y**2) / (1.0**2))

    ks = [1.0, 2.0, 3.6, 5.0]
    phases = [0.0, 0.8, 1.7, 2.6]
    amps = [frag_amp, 0.75 * frag_amp, 0.55 * frag_amp, 0.35 * frag_amp]
    By_fragment = multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=sigma)

    By = By_background + By_fragment
    return Bx, By

def laplacian(F, dx, dy):
    return (
        (np.roll(F, 1, axis=0) - 2 * F + np.roll(F, -1, axis=0)) / dy**2 +
        (np.roll(F, 1, axis=1) - 2 * F + np.roll(F, -1, axis=1)) / dx**2
    )

def cross_sheet_cosine(Bx_hat, By_hat, shift=3):
    Bx_up = np.roll(Bx_hat, -shift, axis=0)
    By_up = np.roll(By_hat, -shift, axis=0)

    Bx_dn = np.roll(Bx_hat, shift, axis=0)
    By_dn = np.roll(By_hat, shift, axis=0)

    cos_map = Bx_up * Bx_dn + By_up * By_dn
    cos_map[:shift, :] = np.nan
    cos_map[-shift:, :] = np.nan
    return cos_map

def failed_mask(cos_map, threshold=THRESHOLD):
    return cos_map < threshold

def activity_map(cos_map, threshold=THRESHOLD):
    act = np.maximum(0.0, threshold - cos_map)
    return np.nan_to_num(act, nan=0.0)

def central_band_mask(mask, Y, half_width=0.8):
    return mask & (np.abs(Y) < half_width)

def connected_components(mask):
    visited = np.zeros_like(mask, dtype=bool)
    rows, cols = mask.shape
    count = 0
    sizes = []

    for i in range(rows):
        for j in range(cols):
            if mask[i, j] and not visited[i, j]:
                count += 1
                q = deque([(i, j)])
                visited[i, j] = True
                size = 0

                while q:
                    r, c = q.popleft()
                    size += 1
                    for rr, cc in [(r-1, c), (r+1, c), (r, c-1), (r, c+1)]:
                        if 0 <= rr < rows and 0 <= cc < cols:
                            if mask[rr, cc] and not visited[rr, cc]:
                                visited[rr, cc] = True
                                q.append((rr, cc))
                sizes.append(size)
    return count, sizes

def estimate_failed_width_vs_x(cos_map, y, threshold=THRESHOLD):
    widths = np.full(cos_map.shape[1], np.nan)
    for j in range(cos_map.shape[1]):
        col = cos_map[:, j]
        bad = np.isfinite(col) & (col < threshold)
        if np.any(bad):
            y_bad = y[bad]
            widths[j] = y_bad.max() - y_bad.min()
    return widths

def local_min_count_1d(arr):
    arr = np.asarray(arr, dtype=float)
    good = np.isfinite(arr)
    idx = np.where(good)[0]
    if len(idx) < 3:
        return 0
    vals = arr[idx]
    count = 0
    for k in range(1, len(vals) - 1):
        if vals[k] < vals[k - 1] and vals[k] < vals[k + 1]:
            count += 1
    return count

## 2. Initial fragmented field

We start from a moderately fragmented sheet, close to the middle regime of Notebook 03.

In [ ]:
delta = 0.35
frag_amp = 0.28
sigma_seed = 0.38

Bx, By = initial_field(X, Y, delta=delta, base_eps=0.06, frag_amp=frag_amp, sigma=sigma_seed)
Bx_target = harris_target(X, Y, delta=delta)

Bx_hat, By_hat = normalize_field(Bx, By)
cos0 = cross_sheet_cosine(Bx_hat, By_hat, shift=3)
mask0 = failed_mask(cos0)

print("initial below-threshold fraction =", np.nanmean(mask0))

In [ ]:
skip = 12
plt.figure(figsize=(8, 5))
plt.quiver(X[::skip, ::skip], Y[::skip, ::skip], Bx_hat[::skip, ::skip], By_hat[::skip, ::skip])
plt.xlabel("x")
plt.ylabel("y")
plt.title("Initial Normalized Magnetic Field")
plt.show()

In [ ]:
plt.figure(figsize=(9, 4.8))
plt.imshow(cos0, extent=[x.min(), x.max(), y.min(), y.max()], origin="lower", aspect="auto")
plt.colorbar(label="cross-sheet cosine")
plt.contour(X, Y, cos0, levels=[THRESHOLD], colors="white", linewidths=1.8, linestyles="--")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Initial Cross-Sheet Cosine Map")
plt.show()

## 3. Pseudo-time update rule

We evolve the fields with:

- diffusion on both components
- restoring drift of \(B_x\) toward the Harris target
- threshold activity concentrated near the sheet
- damping of \(B_y\)

The goal is not full plasma realism. The goal is controlled time evolution of fragmented threshold structure.

In [ ]:
# Dynamics parameters
dt = 0.0018
nu = 0.0012
alpha = 0.25
beta = 0.85
gamma = 0.18
n_steps = 220
save_steps = [0, 20, 50, 100, 160, 220]

# Fixed seed pattern used in the activity-driven update
ks = [1.0, 2.0, 3.6, 5.0]
phases = [0.0, 0.8, 1.7, 2.6]
amps = [frag_amp, 0.75 * frag_amp, 0.55 * frag_amp, 0.35 * frag_amp]
seed_pattern = multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=sigma_seed)

print("dt, nu, alpha, beta, gamma =", dt, nu, alpha, beta, gamma)
print("save steps =", save_steps)

In [ ]:
def one_step(Bx, By, Bx_target, seed_pattern, dt, dx, dy, nu, alpha, beta, gamma):
    Bx_hat, By_hat = normalize_field(Bx, By)
    cos_map = cross_sheet_cosine(Bx_hat, By_hat, shift=3)
    act = activity_map(cos_map, threshold=THRESHOLD)

    Bx_new = Bx + dt * (
        nu * laplacian(Bx, dx, dy)
        - alpha * (Bx - Bx_target)
    )

    By_new = By + dt * (
        nu * laplacian(By, dx, dy)
        + beta * act * seed_pattern
        - gamma * By
    )

    # Soft boundary restoration at top and bottom
    Bx_new[0, :] = Bx_target[0, :]
    Bx_new[-1, :] = Bx_target[-1, :]
    By_new[0, :] = 0.0
    By_new[-1, :] = 0.0

    return Bx_new, By_new, cos_map, act

## 4. Run the time evolution

We track:

- component count in the central band
- width variation of the failed band
- minimum centerline cosine
- centerline local-minimum count
- below-threshold fractions

In [ ]:
snapshots = {}
metrics = {
    "step": [],
    "failed_fraction": [],
    "central_failed_fraction": [],
    "component_count": [],
    "width_mean": [],
    "width_std": [],
    "centerline_min": [],
    "centerline_min_count": [],
}

Bx_t = Bx.copy()
By_t = By.copy()

for step in range(n_steps + 1):
    Bx_hat_t, By_hat_t = normalize_field(Bx_t, By_t)
    cos_map_t = cross_sheet_cosine(Bx_hat_t, By_hat_t, shift=3)
    mask_t = failed_mask(cos_map_t)
    central_t = central_band_mask(mask_t, Y, half_width=0.8)

    comp_count, comp_sizes = connected_components(np.nan_to_num(central_t, nan=False).astype(bool))
    widths_t = estimate_failed_width_vs_x(cos_map_t, y, threshold=THRESHOLD)

    center_row = np.argmin(np.abs(y - 0.0))
    centerline_t = cos_map_t[center_row, :]

    metrics["step"].append(step)
    metrics["failed_fraction"].append(float(np.nanmean(mask_t)))
    metrics["central_failed_fraction"].append(float(np.nanmean(central_t)))
    metrics["component_count"].append(int(comp_count))
    metrics["width_mean"].append(float(np.nanmean(widths_t)) if np.any(np.isfinite(widths_t)) else np.nan)
    metrics["width_std"].append(float(np.nanstd(widths_t)) if np.any(np.isfinite(widths_t)) else np.nan)
    metrics["centerline_min"].append(float(np.nanmin(centerline_t)))
    metrics["centerline_min_count"].append(int(local_min_count_1d(centerline_t)))

    if step in save_steps:
        snapshots[step] = {
            "Bx": Bx_t.copy(),
            "By": By_t.copy(),
            "cos_map": cos_map_t.copy(),
            "mask": mask_t.copy(),
            "central": central_t.copy(),
            "centerline": centerline_t.copy(),
            "widths": widths_t.copy(),
        }

    if step < n_steps:
        Bx_t, By_t, _, _ = one_step(
            Bx_t, By_t, Bx_target, seed_pattern,
            dt=dt, dx=dx, dy=dy,
            nu=nu, alpha=alpha, beta=beta, gamma=gamma
        )

print("done")
print("saved snapshots:", list(snapshots.keys()))

## 5. Snapshot cosine maps

These panels show the time evolution of the 45°-structured shear layer.

In [ ]:
for step in save_steps:
    snap = snapshots[step]
    plt.figure(figsize=(9, 4.8))
    plt.imshow(
        snap["cos_map"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.colorbar(label="cross-sheet cosine")
    plt.contour(
        X, Y, snap["cos_map"],
        levels=[THRESHOLD],
        colors="white",
        linewidths=1.8,
        linestyles="--"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Cross-Sheet Cosine Map (step={step})")
    plt.show()

## 6. Snapshot threshold masks

These show how the below-threshold pockets persist, merge, or relax.

In [ ]:
for step in save_steps:
    snap = snapshots[step]
    plt.figure(figsize=(9, 4.8))
    plt.imshow(
        snap["mask"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Below-Threshold Region (step={step})")
    plt.show()

## 7. Central-band masks

These isolate the most relevant pocket structure near the sheet center.

In [ ]:
for step in save_steps:
    snap = snapshots[step]
    plt.figure(figsize=(9, 4.8))
    plt.imshow(
        snap["central"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Central Failed Region (step={step})")
    plt.show()

## 8. Failed-band width versus x at selected times

This tracks where the sheet narrows or broadens.

In [ ]:
for step in save_steps:
    snap = snapshots[step]
    plt.figure(figsize=(9, 4))
    plt.plot(x, snap["widths"])
    plt.xlabel("x")
    plt.ylabel("failed-band width in y")
    plt.title(f"Failed-Band Width vs x (step={step})")
    plt.show()

## 9. Centerline cosine at selected times

This is the cleanest one-dimensional trace of evolving shear structure.

In [ ]:
for step in save_steps:
    snap = snapshots[step]
    plt.figure(figsize=(9, 4))
    plt.plot(x, snap["centerline"])
    plt.axhline(THRESHOLD, linestyle="--")
    plt.xlabel("x")
    plt.ylabel("cross-sheet cosine at y≈0")
    plt.title(f"Centerline Cosine (step={step})")
    plt.show()

## 10. Time-series metrics

These curves summarize the pseudo-time dynamics.

In [ ]:
steps = np.array(metrics["step"])
failed_fraction = np.array(metrics["failed_fraction"])
central_failed_fraction = np.array(metrics["central_failed_fraction"])
component_count = np.array(metrics["component_count"])
width_mean = np.array(metrics["width_mean"])
width_std = np.array(metrics["width_std"])
centerline_min = np.array(metrics["centerline_min"])
centerline_min_count = np.array(metrics["centerline_min_count"])

plt.figure(figsize=(8, 5))
plt.plot(steps, component_count)
plt.xlabel("step")
plt.ylabel("central component count")
plt.title("Pocket Count vs Time")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, width_std)
plt.xlabel("step")
plt.ylabel("width std along x")
plt.title("Width Variation vs Time")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, centerline_min)
plt.xlabel("step")
plt.ylabel("minimum centerline cosine")
plt.title("Deepest Centerline Shear vs Time")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, centerline_min_count)
plt.xlabel("step")
plt.ylabel("centerline local minima count")
plt.title("Centerline Pocket Count Proxy vs Time")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps, failed_fraction, label="domain-wide below-threshold fraction")
plt.plot(steps, central_failed_fraction, label="central-band below-threshold fraction")
plt.xlabel("step")
plt.ylabel("below-threshold fraction")
plt.title("Below-Threshold Fraction vs Time")
plt.legend()
plt.show()

## 11. Compact snapshot summary

In [ ]:
for step in save_steps:
    idx = metrics["step"].index(step)
    print(
        f"step={step:3d} | "
        f"failed_fraction={metrics['failed_fraction'][idx]:.4f} | "
        f"central_failed_fraction={metrics['central_failed_fraction'][idx]:.4f} | "
        f"component_count={metrics['component_count'][idx]} | "
        f"width_mean={metrics['width_mean'][idx]:.4f} | "
        f"width_std={metrics['width_std'][idx]:.4f} | "
        f"centerline_min={metrics['centerline_min'][idx]:.4f} | "
        f"centerline_min_count={metrics['centerline_min_count'][idx]}"
    )

## 12. Interpretation

Notebook 04 adds a minimal dynamics layer to the repo.

What it shows:

- fragmented shear pockets can be evolved in pseudo-time
- the 45° boundary remains a useful geometric tracker
- pocket counts, width variation, and centerline structure can be monitored as explicit time-series

What it still does **not** claim:

- full resistive or kinetic reconnection physics
- quantitative solar-plasma rates
- a substitute for MHD or PIC simulation

Best interpretation:

> a minimal constraint-relaxation dynamics model for threshold-structured current sheets.

A natural next notebook would introduce stochastic forcing or randomized seed updates, so that one can study distributions of pocket counts and lifetimes rather than only one deterministic trajectory.